# Предсказание рейтинга SoFIFA на следующий год после сезона

Матч → трансформер (обученный энкодер) → эмбеддинг по каждому игроку в матче → **новая голова** предсказывает рейтинг игрока в **следующем году** после сезона (таргет из `dataset/sofifa_ratings_by_season.csv`).

In [1]:
import sys
from pathlib import Path
import pickle

ROOT = Path(".").resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from safetensors.torch import load_file as load_safetensors

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
ENCODER_CKPT_DIR = ROOT / "model_trained" / "embed_128"
METADATA_DIR = ROOT / "notebooks" / "train_sample_processed" / "metadata"
PROCESSED_CSV = ROOT / "notebooks" / "train_sample_processed" / "processed.csv"
RATINGS_BY_SEASON_CSV = ROOT / "dataset" / "sofifa_ratings_by_season.csv"
OUTPUT_DIR = ROOT / "notebooks" / "rating_head_output" / "next_year"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert ENCODER_CKPT_DIR.exists(), f"Чекпоинт не найден: {ENCODER_CKPT_DIR}"
assert METADATA_DIR.exists(), f"Метаданные не найдены: {METADATA_DIR}"
assert PROCESSED_CSV.exists(), f"CSV не найден: {PROCESSED_CSV}"
assert RATINGS_BY_SEASON_CSV.exists(), f"Рейтинги по сезонам не найдены: {RATINGS_BY_SEASON_CSV}"

In [3]:
def load_vocab(metadata_dir: Path):
    with open(metadata_dir / "player_name2id.pickle", "rb") as f:
        player_name2id = pickle.load(f)
    with open(metadata_dir / "team_name2id.pickle", "rb") as f:
        team_name2id = pickle.load(f)
    n = len(player_name2id)
    return {
        "player_name2id": player_name2id,
        "team_name2id": team_name2id,
        "player_pad_token_id": n + 1,
        "team_pad_token_id": len(team_name2id),
    }

vocab = load_vocab(METADATA_DIR)
player_pad_token_id = vocab["player_pad_token_id"]
teams_vocab_size = vocab["team_pad_token_id"] + 1
print("player_pad_token_id:", player_pad_token_id)

player_pad_token_id: 6394


In [4]:
import pandas as pd
from data.dataset import MatchDatasetMPP
from data.sofifa import (
    build_per_match_embeddings_next_year,
    SofifaAggregatedDataset,
)
from models.encoder import PlayerEncoder
from data.preprocessing import FORM_STATS_SIZE

df = pd.read_csv(PROCESSED_CSV)
match_ds = MatchDatasetMPP(
    df,
    player_name2id=vocab["player_name2id"],
    team_name2id=vocab["team_name2id"],
    max_seq_length=36,
    player_pad_token_id=player_pad_token_id,
    team_pad_token_id=vocab["team_pad_token_id"],
)

ratings_by_season = pd.read_csv(RATINGS_BY_SEASON_CSV)
ratings_by_season = ratings_by_season.dropna(subset=["overall"])
ratings_by_season["rating_year"] = ratings_by_season["rating_year"].astype(int)
print("Строк рейтингов по сезонам (с overall):", len(ratings_by_season))

embed_size = 128
encoder = PlayerEncoder(
    embed_size=embed_size,
    num_layers=1,
    heads=2,
    forward_expansion=4,
    dropout=0.05,
    form_stats_size=FORM_STATS_SIZE,
    players_vocab_size=player_pad_token_id,
    teams_vocab_size=teams_vocab_size,
    positions_vocab_size=25,
    use_teams_embeddings=False,
    position_enc_type="learned",
)
state = load_safetensors(ENCODER_CKPT_DIR / "model.safetensors")
enc_state = {k.replace("encoder.", ""): v for k, v in state.items() if k.startswith("encoder.")}
encoder.load_state_dict(enc_state, strict=True)
for p in encoder.parameters():
    p.requires_grad = False

print("Строим эмбеддинги по матчам и таргеты (рейтинг на следующий год)...")
embeddings, overalls, player_names = build_per_match_embeddings_next_year(
    encoder, match_ds, df, ratings_by_season, device
)
print("Сэмплов (игрок-матч с таргетом):", len(overalls))

Строк рейтингов по сезонам (с overall): 2716
Строим эмбеддинги по матчам и таргеты (рейтинг на следующий год)...
Сэмплов (игрок-матч с таргетом): 14151


In [5]:
meta = pd.DataFrame({"overall": overalls})
dataset = SofifaAggregatedDataset(embeddings, meta)
n = len(dataset)
n_val = max(1, int(n * 0.15))
n_train = n - n_val
train_ds = Subset(dataset, range(0, n_train))
eval_ds = Subset(dataset, range(n_train, n))
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
eval_loader = DataLoader(eval_ds, batch_size=64, shuffle=False, num_workers=0)
print("Train:", n_train, "Eval:", n_val)

Train: 12029 Eval: 2122


In [7]:
# Головы AdaBoost и XGBoost на тех же эмбеддингах (train)
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb

X_train = embeddings[:n_train]
y_train = np.array(overalls[:n_train], dtype=np.float64)
X_eval = embeddings[n_train:n]
y_eval = np.array(overalls[n_train:n], dtype=np.float64)

model_adaboost = AdaBoostRegressor(n_estimators=100, learning_rate=0.5, random_state=42)
model_adaboost.fit(X_train, y_train)
rmse_adaboost = np.sqrt(mean_squared_error(y_eval, model_adaboost.predict(X_eval)))
print("AdaBoost eval RMSE:", round(rmse_adaboost, 4))

model_xgboost = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model_xgboost.fit(X_train, y_train)
rmse_xgboost = np.sqrt(mean_squared_error(y_eval, model_xgboost.predict(X_eval)))
print("XGBoost eval RMSE:", round(rmse_xgboost, 4))

AdaBoost eval RMSE: 5.9649
XGBoost eval RMSE: 5.4336


In [8]:
from models.heads import RegressionHead
from training.rating_trainer import RatingHeadTrainer

head = RegressionHead(embed_size=embed_size, output_dim=1, hidden_dim=128, pool="per_sequence")
head = head.to(device)

trainer = RatingHeadTrainer(
    head=head,
    train_loader=train_loader,
    eval_loader=eval_loader,
    output_dir=str(OUTPUT_DIR),
    num_epochs=50,
    lr=1e-3,
    weight_decay=0.01,
    device=device,
    logging_steps=10,
    save_best=True,
)
trainer.train()
print("Голова сохранена:", OUTPUT_DIR / "rating_head.pt")

Epoch 1/50  train_loss=1709.0915  eval_rmse=23.8612
Epoch 10/50  train_loss=34.5840  eval_rmse=7.0202
Epoch 20/50  train_loss=23.4955  eval_rmse=6.3833
Epoch 30/50  train_loss=20.4743  eval_rmse=5.7700
Epoch 40/50  train_loss=18.8871  eval_rmse=5.7370
Epoch 50/50  train_loss=18.0458  eval_rmse=5.7633
Best eval RMSE: 5.611573569040121
Голова сохранена: C:\Users\vasilii\Documents\ML_project-football-\notebooks\rating_head_output\next_year\rating_head.pt
Голова сохранена: C:\Users\vasilii\Documents\ML_project-football-\notebooks\rating_head_output\next_year\rating_head.pt


In [11]:
# Таблица: игрок — настоящий рейтинг — предсказания всех моделей (нейросеть, AdaBoost, XGBoost)
head.eval()
preds_nn = []
with torch.no_grad():
    for idx in range(n_train, n):
        emb = torch.from_numpy(embeddings[idx].copy()).float().unsqueeze(0).unsqueeze(1).to(device)
        mask = torch.ones(1, 1, dtype=torch.long, device=device)
        pred = head(emb, mask).squeeze().cpu().item()
        preds_nn.append(pred)
preds_adaboost = model_adaboost.predict(X_eval)
preds_xgboost = model_xgboost.predict(X_eval)
table = pd.DataFrame({
    "игрок": player_names[n_train:n],
    "настоящий_рейтинг": overalls[n_train:n],
    "предсказание_нейросеть": preds_nn,
    "предсказание_AdaBoost": preds_adaboost,
    "предсказание_XGBoost": preds_xgboost,
})
table

,игрок,настоящий_рейтинг,предсказание_нейросеть,предсказание_AdaBoost,предсказание_XGBoost
0,Andrea Masiello,69.0,66.943382,75.832495,74.655014
1,Marten de Roon,81.0,80.576477,74.749926,73.205177
2,Marcelo Brozović,80.0,82.516594,75.952612,76.934387
3,Nicola Leali,73.0,73.768410,76.193443,74.863914
4,Andrea Belotti,75.0,75.498344,76.057111,77.024971
...,...,...,...,...,...
2117,Chris Mepham,72.0,69.675346,76.427202,77.357628
2118,Joe Rodon,77.0,68.712952,75.840165,75.344841
2119,Ben Davies,75.0,70.884987,76.046810,76.862488
2120,Aaron Ramsey,70.0,71.991486,75.886448,75.597054
